# Notebook 06: Visualization Hub

This notebook centralizes all visualization for the hybrid bias correction
project. It generates diagnostic plots from three domains:

1. **QA Framework** (Steps 1--11e): Eight gridded plot types + four
   station-level breakdowns (region, component×region, province, station)
2. **Taylor Diagrams** (Steps 12--19): Product comparison against BMKG stations
3. **Station Validation** (Steps 20--23): Spatial maps, multi-threshold curves,
   and regional box plots from notebook 05 output

A unified batch cell (Step 24) generates all figures across all 36 dekadal
periods in a single run.

All plot functions are imported from `src/visualisation.py` and
`src/taylor_diagram.py`. Each plot type saves into its own sub-folder
under `figures/`.

## 1 Connect Google Drive (Colab only)

This section is only required when running in **Google Colab** and your project/data are stored in Google Drive.

- Mounting Drive makes your repository and datasets accessible under `/content/drive`.
- If you run this notebook locally (Jupyter / VS Code), **skip this section**.

**Expected structure (Drive)**
After mounting, your project root should contain:
- `notebooks/`
- `src/`
- `config.yml` (or `config.yaml`)

Proceed to the code cell below to mount Drive.


In [ ]:
from google.colab import drive
import os

if os.path.exists("/content/drive"):
    try:
        drive.flush_and_unmount()
    except Exception:
        pass
drive.mount("/content/drive")


**Troubleshooting:**  
- If Colab becomes disconnected, Reconnect the runtime and rerun the mounting cell.
- If we receive an error such as `Mountpoint must not already contain files`, delete all the sub-folders under "/content/drive" from the Files panel before retrying. We need to delete these one by one starting from the innermost folders, until the last "drive" folder is deleted.


## 2 Install packages (only if needed)

In most cases, **Google Colab already includes the packages required** for this workflow. The most common missing dependency is **`netCDF4`** (NetCDF I/O support).

### Check what is already installed (Colab)
Before installing anything, you can inspect the current environment by running `!pip list`.

- If all required packages are present and only `netCDF4` is missing, install **only `netCDF4`**.
- If other required packages are missing from `!pip list`, install them **together with** `netCDF4` in the code cell below.

### Local Jupyter note
If you are running in a **local environment** (Jupyter / VS Code), assume all dependencies were installed when preparing the environment following the **main repository README**. In that case, you can skip this section.

Proceed to the code cell below only when installation is necessary.


In [ ]:
!pip install netCDF4 SkillMetrics


## 3 Quality Assessment Visualization

This section generates spatial maps, distribution plots, and summary charts from
the composite quality indices computed by notebook 04. Eight complementary plot
types are produced, each highlighting a different aspect of correction quality:

1. **CQI spatial maps** -- continuous quality index for each correction method
2. **Categorical quality maps** -- Poor/Fair/Good/Excellent classification
3. **Method improvement map** -- CQI difference (LSEQM+DL minus LS)
4. **Component quality maps** -- basic, distribution, temporal sub-scores
5. **Confidence map** -- assessment reliability
6. **CQI distribution** -- empirical CDF and histogram
7. **Quality category summary** -- grouped bar chart of category percentages
8. **Component box plots** -- score distribution across methods

All plot functions are imported from `src/visualisation.py` and each type
saves into its own sub-folder under `figures/qa/`.

---


### Step 1: Environment Setup


In [ ]:
"""Step 1: Environment setup and configuration."""
import os
import sys
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

# Resolve project root
if os.path.exists("/content/drive/MyDrive/hybrid-bias-correction"):
    project_root = "/content/drive/MyDrive/hybrid-bias-correction"
else:
    project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))

if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.config import initialize_config
initialize_config(os.path.join(project_root, "config.yml"))
from src import config

# Output directories for all figure types
figures_dir = os.path.join(config.output_dir, "figures", "qa")
taylor_output_dir = os.path.join(config.output_dir, "figures", "taylor")
station_viz_dir = os.path.join(config.output_dir, "figures", "station_validation")

for d in [figures_dir, taylor_output_dir, station_viz_dir]:
    os.makedirs(d, exist_ok=True)

print(f"Project root:     {project_root}")
print(f"QA figures dir:   {figures_dir}")
print(f"Taylor dir:       {taylor_output_dir}")
print(f"Station viz dir:  {station_viz_dir}")

### Step 2: Select Period and Quality Mode


In [ ]:
"""Step 2: User inputs for interactive exploration."""

# Select month (1-12) and dekad (1, 2, or 3)
month = 1
dekad = 1

# Quality mode: "qualitysd" (single-dekad aggregated) or "qualityts" (per-year)
quality_prefix = "qualitysd"

print(f"Period: month {month}, dekad {dekad}")
print(f"Quality mode: {quality_prefix}")


### Step 3: Load Quality Data


In [ ]:
"""Step 3: Load QA NetCDF files for all correction methods."""
from src.visualisation import load_quality_data

quality_data = load_quality_data(
    month, dekad,
    quality_prefix=quality_prefix,
    config=config,
)

print(f"Loaded {len(quality_data)} method(s): {list(quality_data.keys())}")


### Step 4: CQI Spatial Maps


In [ ]:
"""Step 4: Continuous Quality Index spatial maps."""
from src.visualisation import plot_cqi_spatial

fig = plot_cqi_spatial(
    quality_data, month, dekad,
    quality_prefix=quality_prefix,
    output_dir=figures_dir,
)


### Step 5: Categorical Quality Maps


In [ ]:
"""Step 5: Categorical quality classification maps."""
from src.visualisation import plot_categorical_spatial

fig = plot_categorical_spatial(
    quality_data, month, dekad,
    quality_prefix=quality_prefix,
    output_dir=figures_dir,
)


### Step 6: Method Improvement Map


In [ ]:
"""Step 6: CQI improvement map (LSEQM+DL minus LS)."""
from src.visualisation import plot_improvement

fig, imp_stats = plot_improvement(
    quality_data, month, dekad,
    quality_prefix=quality_prefix,
    output_dir=figures_dir,
)


### Step 7: Component Quality Maps


In [ ]:
"""Step 7: Component quality maps (basic, distribution, temporal)."""
from src.visualisation import plot_components

fig = plot_components(
    quality_data, month, dekad,
    quality_prefix=quality_prefix,
    output_dir=figures_dir,
)


### Step 8: Confidence Map


In [ ]:
"""Step 8: Confidence level spatial map."""
from src.visualisation import plot_confidence

fig = plot_confidence(
    quality_data, month, dekad,
    quality_prefix=quality_prefix,
    output_dir=figures_dir,
)


### Step 9: CQI Distribution Analysis


In [ ]:
"""Step 9: Empirical CDF and histogram of CQI."""
from src.visualisation import plot_cqi_distribution

fig = plot_cqi_distribution(
    quality_data, month, dekad,
    quality_prefix=quality_prefix,
    output_dir=figures_dir,
)


### Step 10: Quality Category Summary


In [ ]:
"""Step 10: Grouped bar chart of quality category percentages."""
from src.visualisation import plot_category_summary

fig, cat_summary = plot_category_summary(
    quality_data, month, dekad,
    quality_prefix=quality_prefix,
    output_dir=figures_dir,
)


### Step 11: Component Box Plots


In [ ]:
"""Step 11: Component score box plots across methods."""
from src.visualisation import plot_component_boxplots

fig = plot_component_boxplots(
    quality_data, month, dekad,
    quality_prefix=quality_prefix,
    output_dir=figures_dir,
)


### Step 11b: QA at Station Locations — Regional Summary

Extract gridded QA values at BMKG station locations, then visualize
median CQI grouped by the 7 island regions (one bar per method).

In [ ]:
"""Step 11b: QA CQI by region — grouped bar chart."""
from src.station_density import load_station_locations
from src.visualisation import plot_qa_regional_bars

station_df = load_station_locations(config.STATION_FILE)

fig = plot_qa_regional_bars(
    quality_data, station_df, month, dekad,
    quality_prefix=quality_prefix,
    output_dir=figures_dir,
)

### Step 11c: QA Components by Region

Box plots of the four QA component scores (CQI, basic statistical,
distribution, temporal) at station locations, grouped by island region.
Uses the best method (LSEQM+DL) only.

In [ ]:
"""Step 11c: QA component box plots by region."""
from src.visualisation import plot_qa_component_by_region

fig = plot_qa_component_by_region(
    quality_data, station_df, month, dekad,
    quality_prefix=quality_prefix,
    output_dir=figures_dir,
)

### Step 11d: QA CQI by Province

Horizontal bar chart of median CQI per province (best method),
colour-coded by parent island region.

In [ ]:
"""Step 11d: QA CQI by province."""
from src.visualisation import plot_qa_province_bars

fig = plot_qa_province_bars(
    quality_data, station_df, month, dekad,
    quality_prefix=quality_prefix,
    output_dir=figures_dir,
)

### Step 11e: QA CQI per Station

Per-station CQI bar chart (best method), filtered by region.
Change `region_filter` to view a different island group, or set to
`None` to show all stations.

In [ ]:
"""Step 11e: QA CQI per station (single region)."""
from src.visualisation import plot_qa_station_bars

fig = plot_qa_station_bars(
    quality_data, station_df, month, dekad,
    quality_prefix=quality_prefix,
    region_filter="Jawa",  # Change to view other regions, or None for all
    output_dir=figures_dir,
)

## 4 Taylor Diagram Analysis

This notebook generates Taylor diagrams that visually summarize the performance
of bias-corrected precipitation products against independent BMKG station
observations.

### What is a Taylor Diagram?

A Taylor diagram (Taylor, 2001) simultaneously displays three statistics in a
single polar plot:

- **Pearson correlation** (angular axis) -- linear agreement with reference
- **Standard deviation ratio** (radial axis) -- variability match
- **Centered RMSE** (distance from reference point) -- overall error

Each product appears as a marker; perfect agreement places the marker on the
reference point (correlation = 1, normalized std = 1). Progressive improvement
from raw satellite to fully corrected product is immediately visible as markers
move toward the reference.

### Products Compared

| Product | Description |
|---------|-------------|
| CPC-UNI | Gauge-based reference (0.5\u00b0, used as correction target) |
| IMERG-L | Raw satellite precipitation (uncorrected) |
| IMERG-F | Gauge-adjusted satellite precipitation (uncorrected) |
| LS | Linear Scaling corrected |
| LSEQM | LS + EQM with GPD tail adjustment |
| LSEQM+DL | Full hybrid framework (LS + EQM + GPD + CNN) |

### Spatial Aggregation Levels

1. **Domain-wide** -- all 171+ BMKG stations pooled (for the paper)
2. **By island group** -- 7 regions (Sumatra, Jawa, Kalimantan, etc.)
3. **By province** -- individual provinces (min 3 stations)
4. **By station** -- individual station markers with domain aggregate overlay

### Prerequisites

- Notebook 02 must have completed (corrected files for LS, LSEQM, LSEQMDL)
- BMKG station data CSV must be available (location + observations)
- CPC-UNI and IMERG-L files must be available

### Reference

Taylor, K. E. (2001). Summarizing multiple aspects of model performance in a
single diagram. *Journal of Geophysical Research*, 106(D7), 7183--7192.

### Step 12: Compute Taylor Diagram Statistics

This step processes all **36 dekads** (12 months &times; 3 dekads), loading
six gridded products (CPC, IMERG-L, IMERG-F, LS, LSEQM, LSEQM+DL) for
each dekad and extracting values at each BMKG station location.

Running statistics (sums, sums of squares, cross-products) are accumulated
per station per product, enabling exact computation of correlation, standard
deviation, and RMSE without keeping all raw data in memory.

**Expected runtime**: 3--10 minutes depending on disk I/O speed.

In [ ]:
"""
Step 12: Compute Taylor statistics across all 36 dekads.

This is the most time-consuming step. The resulting accumulator object
is reused by all subsequent plotting steps.
"""
from src.taylor_diagram import compute_all_taylor_stats

acc, station_locs = compute_all_taylor_stats(config, progress=True)

print(f"\nAccumulator products: {acc.product_keys}")
print(f"Stations: {acc.n_stations}")
print(f"Total paired observations per product:")
for i, key in enumerate(acc.product_keys):
    total_n = acc.n[i].sum()
    active_stations = (acc.n[i] > 0).sum()
    print(f"  {key:>10s}: {total_n:>12,d} pairs across {active_stations} stations")

### Step 13: Domain-Wide Taylor Diagram

Pool all station data across the full Indonesian domain. This produces one
marker per product, showing the overall performance comparison.

**This is the figure intended for the paper** (Figure 4 in revision notes).

In [ ]:
"""
Step 13: Generate domain-wide Taylor diagram.
"""
from src.taylor_diagram import generate_domain_taylor, print_taylor_stats

fig_domain = generate_domain_taylor(
    acc, station_locs,
    output_dir=taylor_output_dir,
    normalize=True,
    max_std_ratio=2.0,
)
plt.show()

### Step 14: Taylor Diagrams by Island Group

Generate a multi-panel figure with one Taylor diagram per major island
group: Sumatra, Jawa, Kalimantan, Sulawesi, Bali Nusa Tenggara, Maluku,
and Papua. Stations are pooled within each region.

This reveals regional differences in correction performance (e.g., Java
with dense gauge coverage vs. Papua with sparse coverage).

In [ ]:
"""
Step 14: Generate per-island Taylor diagrams.
"""
from src.taylor_diagram import generate_island_taylor

fig_island = generate_island_taylor(
    acc, station_locs,
    output_dir=taylor_output_dir,
    normalize=True,
)
plt.show()

### Step 15: Taylor Diagrams by Province

Generate per-province Taylor diagrams for provinces with at least 3
stations. Provinces with fewer stations are skipped as the pooled
statistics would be unreliable.

This provides the finest administrative-level breakdown of correction
performance.

In [ ]:
"""
Step 15: Generate per-province Taylor diagrams.
"""
from src.taylor_diagram import generate_province_taylor

fig_province = generate_province_taylor(
    acc, station_locs,
    output_dir=taylor_output_dir,
    min_stations=3,
    normalize=True,
)
plt.show()

### Step 16: Station-Level Taylor Diagram

Shows individual station markers (small, semi-transparent) for each
product, overlaid with larger opaque markers representing the domain-wide
aggregate. This reveals the spread of per-station performance within each
product category.

Stations that cluster tightly around the aggregate marker indicate
consistent correction quality across the network. Wide scatter indicates
spatially variable performance.

In [ ]:
"""
Step 16: Generate station-level Taylor diagram.
"""
from src.taylor_diagram import generate_station_taylor

fig_station = generate_station_taylor(
    acc, station_locs,
    output_dir=taylor_output_dir,
    normalize=True,
    max_std_ratio=2.0,
)
plt.show()

### Step 17: Save Statistics to CSV

Export per-station Taylor statistics for all products to a CSV file.
This enables further analysis (e.g., in R, Excel) without re-running
the computation.

In [ ]:
"""
Step 17: Save per-station Taylor statistics to CSV.
"""
from src.taylor_diagram import save_taylor_stats_csv

csv_path = os.path.join(taylor_output_dir, 'taylor_statistics_per_station.csv')
stats_df = save_taylor_stats_csv(acc, station_locs, csv_path)

print(f"\nCSV shape: {stats_df.shape}")
print(f"Columns: {list(stats_df.columns)}")
stats_df.head(10)

### Step 18: One-Click Generation (Alternative)

If you want to generate everything in a single call instead of running
Steps 12--17 individually, use `generate_all_taylor_diagrams()`. This
computes statistics and produces all four diagram variants plus the CSV.

**Note**: Only run this if you have NOT already run Steps 12--17 above,
as it will repeat the computation.

In [ ]:
"""
Step 18 (Alternative): Generate all Taylor diagrams in one call.

Uncomment and run this cell INSTEAD of Steps 12-17 if preferred.
"""
# from src.taylor_diagram import generate_all_taylor_diagrams
#
# acc, station_locs = generate_all_taylor_diagrams(
#     config,
#     output_dir=taylor_output_dir,
# )

### Step 19: Custom Taylor Diagram (On-Demand)

Generate a Taylor diagram for a specific subset of products or a specific
region. Modify the parameters below as needed.

**Examples:**
- Show only corrected products (exclude raw IMERG)
- Focus on a single island
- Compare only LSEQM vs LSEQM+DL

In [ ]:
"""
Step 19: Custom Taylor diagram for a specific region or product subset.

Modify TARGET_REGION and PRODUCTS_TO_SHOW as needed.
"""
from src.taylor_diagram import plot_taylor_diagram, print_taylor_stats

# --- User settings ---
TARGET_REGION = 'Jawa'  # One of: Sumatra, Jawa, Kalimantan, Sulawesi,
                        #         Bali Nusa Tenggara, Maluku, Papua
                        # Set to None for domain-wide

PRODUCTS_TO_SHOW = None  # None = all products, or e.g. ['ls', 'lseqm', 'lseqmdl']

# --- Compute stats ---
if TARGET_REGION is not None:
    mask = (station_locs['Region'] == TARGET_REGION).values
    n_st = mask.sum()
    stats = acc.compute(station_mask=mask)
    title = f'Taylor Diagram: {TARGET_REGION} ({n_st} stations)'
else:
    stats = acc.compute()
    title = 'Taylor Diagram: Domain-wide'

# --- Print table ---
print_taylor_stats(stats, title=title)

# --- Plot ---
fig, ax = plot_taylor_diagram(
    stats,
    title=title,
    normalize=True,
    products_to_show=PRODUCTS_TO_SHOW,
    max_std_ratio=2.0,
)
plt.show()

## 5 Station Validation Visualization

This section visualizes the independent station validation results from
notebook 05. Three plot types are generated from the CSV outputs:

1. **Station metric scatter maps** -- spatial distribution of 6 key metrics
2. **WMO multi-threshold curves** -- categorical skill across intensity thresholds
3. **Regional box plots** -- metric distributions by main island

All plot functions are imported from `src/visualisation.py` and save into
sub-folders under `figures/station_validation/`.

**Prerequisites**: Notebook 05 batch must have completed (CSV outputs in
`data/output/station_validation/`).

### Step 20: Load Station Validation CSV

Load per-station validation metrics and multi-threshold summaries from
the CSV files produced by notebook 05. Set `month` and `dekad` to match
the period selected in Step 2.

In [ ]:
"""Step 20: Load station validation CSV data for a single period."""
from src.station_density import load_station_locations
from src.station_validation import merge_station_metadata

# Load station locations
station_df = load_station_locations(config.STATION_FILE)

# Build path to station validation CSV
sv_dir = getattr(config, "STATION_VALIDATION_OUTPUT_DIR", None)
if sv_dir is None:
    sv_dir = os.path.join(config.output_dir, "station_validation")

month_str = f"{month:02d}"
dekad_map = {1: "01", 2: "11", 3: "21"}
dekad_str = dekad_map[dekad]

# Load metrics for each method
sv_metrics = {}
for method_key, method_abbr in [("LS", "ls"), ("LSEQM", "lseqm"), ("LSEQMDL", "lseqmdl")]:
    csv_path = os.path.join(
        sv_dir,
        f"station_validation_{method_abbr}_month{month_str}_dekad{dekad_str}.csv",
    )
    if os.path.exists(csv_path):
        sv_metrics[method_key] = pd.read_csv(csv_path, index_col=0)
        print(f"  {method_key}: {len(sv_metrics[method_key])} stations from {csv_path}")
    else:
        print(f"  {method_key}: not found -- {csv_path}")

# Load multi-threshold summaries
sv_mt_summaries = {}
for method_key, method_abbr in [("LS", "ls"), ("LSEQM", "lseqm"), ("LSEQMDL", "lseqmdl")]:
    mt_csv = os.path.join(
        sv_dir,
        f"multi_threshold_summary_{method_abbr}_month{month_str}_dekad{dekad_str}.csv",
    )
    if os.path.exists(mt_csv):
        sv_mt_summaries[method_key] = pd.read_csv(mt_csv, index_col=0)

print(f"\nMetrics loaded: {list(sv_metrics.keys())}")
print(f"Multi-threshold summaries: {list(sv_mt_summaries.keys())}")

### Step 21: Station Metric Scatter Maps

Spatial scatter plots showing the distribution of 6 key validation metrics
(correlation, NSE, relative bias, RMSE, CSI, POD) at each BMKG station
location for the best correction method.

In [ ]:
"""Step 21: Station metric scatter maps."""
from src.visualisation import plot_station_metric_maps

# Use best method available
sv_best = "LSEQMDL" if "LSEQMDL" in sv_metrics else list(sv_metrics.keys())[-1]

if sv_metrics:
    fig = plot_station_metric_maps(
        sv_metrics[sv_best], station_df, month, dekad,
        method_name=sv_best,
        output_dir=station_viz_dir,
    )
else:
    print("No station validation data available.")

### Step 22: Multi-Threshold Performance Curves

WMO/TD-No. 1485 style performance curves showing how categorical verification
scores (POD, FAR, CSI, FBI, ETS, HSS) change across 7 precipitation intensity
thresholds (1--150 mm/day). Each line represents a correction method.

In [ ]:
"""Step 22: WMO multi-threshold performance curves."""
from src.visualisation import plot_multi_threshold_curves

if sv_mt_summaries:
    fig = plot_multi_threshold_curves(
        sv_mt_summaries, month, dekad,
        output_dir=station_viz_dir,
    )
else:
    print("No multi-threshold summaries available.")

### Step 23: Regional Box Plots

Box plots of key validation metrics grouped by main island (Region).
Reveals regional differences in correction quality — e.g., densely
gauged Java vs. sparsely gauged Papua.

In [ ]:
"""Step 23: Regional box plots of station validation metrics."""
from src.visualisation import plot_regional_boxplots
from src.station_validation import merge_station_metadata

if sv_metrics:
    sv_best = "LSEQMDL" if "LSEQMDL" in sv_metrics else list(sv_metrics.keys())[-1]
    regional_df = merge_station_metadata(sv_metrics[sv_best], station_df)
    if "Region" in regional_df.columns:
        fig = plot_regional_boxplots(
            regional_df, month, dekad,
            method_name=sv_best, group_col="Region",
            output_dir=station_viz_dir,
        )
    else:
        print("Region column not found in station metadata.")
else:
    print("No station validation data available.")

---

## Summary

This notebook generated diagnostic visualizations from three complementary
evaluation domains:

1. **QA Framework** (Steps 4--11e): 8 gridded plot types + 4 station-level
   regional/province/station breakdowns from composite quality indices
2. **Taylor Diagrams** (Steps 12--19): Product comparison against BMKG stations
3. **Station Validation** (Steps 20--23): Spatial maps, threshold curves,
   and regional box plots from independent validation

All figures are saved into organized sub-folders under `figures/`.
The unified batch below generates all figures for all 36 periods.

---

### Step 24: Unified Batch -- All Visualization

Generate all figures across all 36 dekadal periods in a single run.
This calls four batch orchestrators:

1. `run_qa_batch_viz()` — 8 QA plot types × 36 periods × 2 quality modes
2. `run_qa_regional_batch_viz()` — QA regional/province/station plots × 36 periods × 2 modes
3. `generate_all_taylor_diagrams()` — 4 Taylor diagram variants + CSV
4. `run_station_validation_batch_viz()` — 3 station validation plot types × 36 periods

> **Prerequisites**: Only **Step 1** (setup) is required.
> Steps 2--23 can be skipped for batch-only runs.

In [ ]:
"""Step 24: Unified batch -- all visualization for all 36 periods."""
from src.visualisation import (
    run_qa_batch_viz,
    run_qa_regional_batch_viz,
    run_station_validation_batch_viz,
)
from src.taylor_diagram import generate_all_taylor_diagrams

# --- 1. QA Framework plots (8 types x 36 periods x 2 modes) ---
print("=" * 60)
print("BATCH 1/4: QA Framework Visualization")
print("=" * 60)
qa_summaries = {}
for _qp in ["qualitysd", "qualityts"]:
    print(f"\n--- Quality mode: {_qp} ---")
    qa_summaries[_qp] = run_qa_batch_viz(
        quality_prefix=_qp,
        output_dir=figures_dir,
        config=config,
        progress=True,
    )

# --- 2. QA Regional / Province / Station plots (x 2 modes) ---
print("\n" + "=" * 60)
print("BATCH 2/4: QA Regional Breakdown")
print("=" * 60)
qa_regional_summaries = {}
for _qp in ["qualitysd", "qualityts"]:
    print(f"\n--- Quality mode: {_qp} ---")
    qa_regional_summaries[_qp] = run_qa_regional_batch_viz(
        quality_prefix=_qp,
        output_dir=figures_dir,
        config=config,
        progress=True,
    )

# --- 3. Taylor Diagrams ---
print("\n" + "=" * 60)
print("BATCH 3/4: Taylor Diagram Generation")
print("=" * 60)
try:
    td_acc, td_locs = generate_all_taylor_diagrams(
        config,
        output_dir=taylor_output_dir,
    )
    print("Taylor diagrams generated successfully.")
except Exception as e:
    print(f"Taylor diagram generation failed: {e}")

# --- 4. Station Validation plots (3 types x 36 periods) ---
print("\n" + "=" * 60)
print("BATCH 4/4: Station Validation Visualization")
print("=" * 60)
sv_summary = run_station_validation_batch_viz(
    output_dir=station_viz_dir,
    config=config,
    progress=True,
)

print("\n" + "=" * 60)
print("ALL BATCHES COMPLETE")
print("=" * 60)

### Step 25: Batch Summary

In [ ]:
"""Step 25: Print aggregated statistics from all batch runs."""
from src.visualisation import print_batch_summary, print_station_validation_viz_summary

print("=== QA Framework Batch ===")
for _qp, _summary in qa_summaries.items():
    print(f"\n--- {_qp} ---")
    print_batch_summary(_summary)

print("\n=== Station Validation Batch ===")
print_station_validation_viz_summary(sv_summary)

---

## End of Code